<a href="https://colab.research.google.com/github/hbanerjee74/litellm-playground/blob/main/OpenHands_HelloWorld.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openhands-sdk

In [ ]:
!pip install litellm

In [ ]:
!pip install boto3

# OpenHands Test

## *OpenAI*

In [ ]:
import os
from google.colab import userdata
from openhands.sdk import LLM, Agent, Conversation

llm = LLM(
    model="gpt-4o",
    api_key=userdata.get('OPENAI_API_KEY'),
    base_url="https://api.openai.com/v1",
)

agent = Agent(
    llm=llm
)

conversation = Conversation(agent=agent, workspace=os.getcwd())
conversation.send_message("Hello!")
conversation.run()

## *Anthropic*

In [ ]:
import os
from google.colab import userdata
from openhands.sdk import LLM, Agent, Conversation

llm = LLM(
    model="claude-sonnet-4-5",
    api_key=userdata.get('ANTHROPIC_API_KEY'),
    base_url="https://api.anthropic.com",
)

agent = Agent(
    llm=llm
)

conversation = Conversation(agent=agent, workspace=os.getcwd())
conversation.send_message("Hello!")
conversation.run()

## *OpenRouter*

In [ ]:
import os
from google.colab import userdata
from openhands.sdk import LLM, Agent, Conversation

llm = LLM(
    model="minimax/minimax-m3",
    api_key=userdata.get('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
)

agent = Agent(
    llm=llm
)

conversation = Conversation(agent=agent, workspace=os.getcwd())
conversation.send_message("Hello!")
conversation.run()

## *Azure Foundry*

In [ ]:
import os
from google.colab import userdata
from openhands.sdk import LLM, Agent, Conversation

llm = LLM(
      model="azure_ai/My-Special-Kimi",
      api_key=userdata.get('AZURE_API_KEY'),
      base_url="https://openhands-test.services.ai.azure.com/",
      api_version="2025-04-01-preview", # Responses API requires >= 2025-03-01-preview
)

agent = Agent(
    llm=llm
)

conversation = Conversation(agent=agent, workspace=os.getcwd())
conversation.send_message("Hello!")
conversation.run()

In [ ]:
import os
from google.colab import userdata
from openhands.sdk import LLM, Agent, Conversation

llm = LLM(
      model="azure_ai/my-claude-sonnet-4-5",
      api_key=userdata.get('AZURE_ANTHROPIC_API_KEY'),
      base_url="https://hb-1959-resource.services.ai.azure.com/",
      api_version="2025-04-01-preview", # Responses API requires >= 2025-03-01-preview
)

agent = Agent(
    llm=llm
)

conversation = Conversation(agent=agent, workspace=os.getcwd())
conversation.send_message("Hello!")
conversation.run()

## *AWS Bedrock*

In [12]:
import openhands.sdk.llm.llm as oh_llm

ARN = "arn:aws:bedrock:ap-southeast-1:825820235435:application-inference-profile/i8eydrwrd1gm"
def _inject(orig):
    def _wrap(*a, **k):
        k["model_id"] = ARN
        k.pop("max_tokens", None); k.pop("max_completion_tokens", None)
        return orig(*a, **k)
    return _wrap

for n in ("litellm_completion", "litellm_acompletion"):
    if hasattr(oh_llm, n):
        setattr(oh_llm, n, _inject(getattr(oh_llm, n)))

In [ ]:
import os, litellm
from google.colab import userdata
from openhands.sdk import LLM, Agent, Conversation
from pydantic import SecretStr

litellm._turn_on_debug()
class BedrockLLM(LLM):
    """Bedrock-only LLM that allows api_key to flow through as the bearer token.

    Overrides the parent's defensive filter (llm.py:1513), which zeros api_key
    for all Bedrock calls. Safe here because this subclass is only used for
    Bedrock — no risk of a non-Bedrock key leaking through.
    """

    def _get_litellm_api_key_value(self) -> str | None:
        if self.api_key:
            assert isinstance(self.api_key, SecretStr)
            return self.api_key.get_secret_value()
        return None

llm = BedrockLLM(
    model="bedrock/converse/amazon.nova-2-lite-v1:0",
    api_key=SecretStr(userdata.get("AWS_BEARER_TOKEN_BEDROCK")),
    aws_region_name="ap-southeast-1",
    reasoning_effort="high",
    max_output_tokens=None,
)

agent = Agent(llm=llm)
conversation = Conversation(agent=agent, workspace=os.getcwd())
conversation.send_message("Hello!")
conversation.run()